In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print("-", filename)

Saving 310104.xlsx to 310104.xlsx
Saving 634501.xlsx to 634501.xlsx
Saving 850109.xlsx to 850109.xlsx
Saving 5206025_SFD_Summary.xlsx to 5206025_SFD_Summary.xlsx
Saving 5682014.xlsx to 5682014.xlsx
Saving 6202012.xlsx to 6202012.xlsx
Saving 6401017.xlsx to 6401017.xlsx
Saving f01hist.xlsx to f01hist.xlsx
Uploaded files:
- 310104.xlsx
- 634501.xlsx
- 850109.xlsx
- 5206025_SFD_Summary.xlsx
- 5682014.xlsx
- 6202012.xlsx
- 6401017.xlsx
- f01hist.xlsx


In [ ]:
!ls /content

310104.xlsx		  5682014.xlsx	634501.xlsx   850109.xlsx   sample_data
5206025_SFD_Summary.xlsx  6202012.xlsx	6401017.xlsx  f01hist.xlsx


In [ ]:
RAW_FOLDER = "/content"

FILES = {
    "retail": "850109.xlsx",
    "unemployment": "6202012.xlsx",
    "cpi": "6401017.xlsx",
    "cash_rate": "f01hist.xlsx",
    "wpi": "634501.xlsx",
    "sfd": "5206025_SFD_Summary.xlsx",
    "population": "310104.xlsx",
    "household_spending": "5682014.xlsx",
}

OUTPUT_FILE = "quarterly_regression_merged_dataset_rebuilt_fullhistory_pop_hs.xlsx"

def path_for(key):
    return os.path.join(RAW_FOLDER, FILES[key])

print("File path setup complete.")

for key, filename in FILES.items():
    print(key, "->", path_for(key))

File path setup complete.
retail -> /content/850109.xlsx
unemployment -> /content/6202012.xlsx
cpi -> /content/6401017.xlsx
cash_rate -> /content/f01hist.xlsx
wpi -> /content/634501.xlsx
sfd -> /content/5206025_SFD_Summary.xlsx
population -> /content/310104.xlsx
household_spending -> /content/5682014.xlsx


In [ ]:
missing_files = []

for key, filename in FILES.items():
    file_path = path_for(key)

    if not os.path.exists(file_path):
        missing_files.append(file_path)

if missing_files:
    print("Missing files found:")
    for file_path in missing_files:
        print("-", file_path)

    raise FileNotFoundError("Please upload the missing files and run again.")
else:
    print("All required files are available.")

All required files are available.


In [ ]:
!ls /content

310104.xlsx		  5682014.xlsx	634501.xlsx   850109.xlsx   sample_data
5206025_SFD_Summary.xlsx  6202012.xlsx	6401017.xlsx  f01hist.xlsx


In [ ]:
STATE_NAMES = {
    "NSW": "New South Wales",
    "VIC": "Victoria",
    "QLD": "Queensland",
    "SA": "South Australia",
    "WA": "Western Australia",
    "TAS": "Tasmania",
    "NT": "Northern Territory",
    "ACT": "Australian Capital Territory",
}

STATE_ORDER = ["ACT", "NSW", "NT", "QLD", "SA", "TAS", "VIC", "WA"]

HS_STATE_ORDER = ["NSW", "QLD", "SA", "TAS", "VIC", "WA"]

print("State setup complete.")

State setup complete.


In [ ]:
def clean_text(x):
    """Standardise spacing in ABS/RBA metadata text."""
    return re.sub(r"\s+", " ", str(x)).strip()


def read_raw_sheet(file_path, sheet_name):
    """Read one raw Excel sheet with no header."""
    return pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None,
        engine="openpyxl"
    )


def get_abs_metadata(df):
    """
    Build a metadata table from a standard ABS spreadsheet.
    """
    meta = []

    for col in range(1, df.shape[1]):
        meta.append({
            "col": col,
            "description": clean_text(df.iat[0, col]),
            "unit": clean_text(df.iat[1, col]) if df.shape[0] > 1 else "",
            "series_type": clean_text(df.iat[2, col]) if df.shape[0] > 2 else "",
            "data_type": clean_text(df.iat[3, col]) if df.shape[0] > 3 else "",
            "frequency": clean_text(df.iat[4, col]) if df.shape[0] > 4 else "",
            "series_id": clean_text(df.iat[9, col]) if df.shape[0] > 9 else "",
        })

    return pd.DataFrame(meta)


def find_first_data_row(df):
    """Find the first row where column A contains a valid date."""
    dates = pd.to_datetime(df.iloc[:, 0], errors="coerce")
    valid_dates = dates[dates.notna()]

    if valid_dates.empty:
        raise ValueError("Could not find any date rows in column A.")

    return valid_dates.index[0]


def add_quarter_fields(data, date_col="Date"):
    """Add Quarter and Quarter_End columns from a date column."""
    data["Quarter"] = data[date_col].dt.to_period("Q").astype(str)
    data["Quarter_End"] = data[date_col].dt.to_period("Q").dt.end_time.dt.normalize()
    return data


def wide_state_series_to_long(df, selected_cols, value_name):
    """
    Convert selected state columns from a wide ABS sheet into long panel format.
    selected_cols format:
    [(state_abbreviation, column_number), ...]
    """
    start_row = find_first_data_row(df)
    dates = pd.to_datetime(df.iloc[start_row:, 0], errors="coerce")

    frames = []

    for state, col in selected_cols:
        values = pd.to_numeric(df.iloc[start_row:, col], errors="coerce")

        temp = pd.DataFrame({
            "Date": dates.values,
            "State": state,
            value_name: values.values,
        })

        frames.append(temp)

    out = pd.concat(frames, ignore_index=True)
    out = out.dropna(subset=["Date"])
    out = add_quarter_fields(out, "Date")

    return out.drop(columns=["Date"])


def wide_national_series_to_quarterly(df, selected_col, value_name):
    """Extract a quarterly national ABS series."""
    start_row = find_first_data_row(df)

    out = pd.DataFrame({
        "Date": pd.to_datetime(df.iloc[start_row:, 0], errors="coerce"),
        value_name: pd.to_numeric(df.iloc[start_row:, selected_col], errors="coerce"),
    })

    out = out.dropna(subset=["Date"])
    out = add_quarter_fields(out, "Date")

    return out.drop(columns=["Date"])


def monthly_national_to_quarter_average(df, selected_col, value_name):
    """Convert monthly national data to quarterly average."""
    start_row = find_first_data_row(df)

    monthly = pd.DataFrame({
        "Date": pd.to_datetime(df.iloc[start_row:, 0], errors="coerce"),
        value_name: pd.to_numeric(df.iloc[start_row:, selected_col], errors="coerce"),
    }).dropna(subset=["Date"])

    monthly["Quarter"] = monthly["Date"].dt.to_period("Q").astype(str)

    quarterly = monthly.groupby("Quarter", as_index=False)[value_name].mean()
    quarterly["Quarter_End"] = (
        pd.PeriodIndex(quarterly["Quarter"], freq="Q")
        .end_time
        .normalize()
    )

    return quarterly[["Quarter", "Quarter_End", value_name]]


def monthly_state_series_to_quarter_average(df, selected_cols, value_name):
    """
    Convert selected monthly state series to quarterly averages.
    selected_cols format:
    [(state_abbreviation, series_type_used, column_number), ...]
    """
    start_row = find_first_data_row(df)
    dates = pd.to_datetime(df.iloc[start_row:, 0], errors="coerce")

    frames = []

    for state, series_type, col in selected_cols:
        values = pd.to_numeric(df.iloc[start_row:, col], errors="coerce")

        temp = pd.DataFrame({
            "Date": dates.values,
            "State": state,
            value_name: values.values,
            "Unemployment_Series_Type": series_type,
        })

        temp = temp.dropna(subset=["Date"])
        temp["Quarter"] = temp["Date"].dt.to_period("Q").astype(str)

        frames.append(temp)

    long_data = pd.concat(frames, ignore_index=True)

    quarterly = (
        long_data
        .groupby(["State", "Unemployment_Series_Type", "Quarter"], as_index=False)[value_name]
        .mean()
    )

    quarterly["Quarter_End"] = (
        pd.PeriodIndex(quarterly["Quarter"], freq="Q")
        .end_time
        .normalize()
    )

    return quarterly[[
        "Quarter",
        "Quarter_End",
        "State",
        value_name,
        "Unemployment_Series_Type",
    ]]


def select_state_columns(
    df,
    description_keyword,
    series_type=None,
    exclude_keywords=None,
    states=STATE_NAMES,
    extra_required_keywords=None,
):
    """Select one matching ABS column per state."""
    meta = get_abs_metadata(df)
    selected = []

    for abbr, full_name in states.items():
        mask = meta["description"].str.contains(
            description_keyword,
            case=False,
            regex=False
        )

        mask &= meta["description"].str.contains(
            full_name,
            case=False,
            regex=False
        )

        if series_type is not None:
            mask &= meta["series_type"].eq(series_type)

        if extra_required_keywords:
            for kw in extra_required_keywords:
                mask &= meta["description"].str.contains(
                    kw,
                    case=False,
                    regex=False
                )

        if exclude_keywords:
            for kw in exclude_keywords:
                mask &= ~meta["description"].str.contains(
                    kw,
                    case=False,
                    regex=False
                )

        candidates = meta[mask]

        if candidates.empty:
            raise ValueError(
                f"No matching column found for state={abbr}, "
                f"description_keyword={description_keyword}, "
                f"series_type={series_type}"
            )

        selected.append((abbr, int(candidates.iloc[0]["col"])))

    return selected


def select_national_column(df, required_keywords, series_type=None):
    """Select one national ABS column by searching the metadata descriptions."""
    meta = get_abs_metadata(df)

    mask = pd.Series(True, index=meta.index)

    for text in required_keywords:
        mask &= meta["description"].str.contains(
            text,
            case=False,
            regex=False
        )

    if series_type is not None:
        mask &= meta["series_type"].eq(series_type)

    candidates = meta[mask]

    if candidates.empty:
        raise ValueError(
            f"No national column found for {required_keywords}, {series_type}"
        )

    return int(candidates.iloc[0]["col"])


def find_rba_cash_rate_column(df):
    """Find the RBA column where Title = Cash Rate Target."""
    first_col = df.iloc[:, 0].astype(str).str.strip()

    if not first_col.eq("Title").any():
        raise ValueError("Could not find the 'Title' metadata row in RBA cash rate file.")

    title_row = first_col[first_col.eq("Title")].index[0]

    for col in range(1, df.shape[1]):
        if str(df.iat[title_row, col]).strip() == "Cash Rate Target":
            return col

    raise ValueError("Could not find the Cash Rate Target column in f01hist.xlsx.")


print("All helper functions created successfully.")

All helper functions created successfully.


In [ ]:
retail_raw = read_raw_sheet(path_for("retail"), "Data1")

retail_cols = select_state_columns(
    retail_raw,
    description_keyword="Turnover - Chain Volume Measure",
    series_type="Seasonally Adjusted",
)

retail_long = wide_state_series_to_long(
    retail_raw,
    selected_cols=retail_cols,
    value_name="Retail_Turnover_CVM_SA",
)

print("Step 8 complete: Retail turnover extracted.")
print(retail_long.shape)
retail_long.head()

Step 8 complete: Retail turnover extracted.
(1344, 4)


,State,Retail_Turnover_CVM_SA,Quarter,Quarter_End
0,NSW,9904.6,1983Q3,1983-09-30
1,NSW,10028.8,1983Q4,1983-12-31
2,NSW,10012.0,1984Q1,1984-03-31
3,NSW,9989.5,1984Q2,1984-06-30
4,NSW,10091.4,1984Q3,1984-09-30


In [ ]:
population_raw = read_raw_sheet(path_for("population"), "Data1")

population_cols = select_state_columns(
    population_raw,
    description_keyword="Estimated Resident Population ; Persons",
    series_type="Original",
)

population_long = wide_state_series_to_long(
    population_raw,
    selected_cols=population_cols,
    value_name="Population",
)

print("Step 9 complete: Population extracted.")
print(population_long.shape)
population_long.head()

Step 9 complete: Population extracted.
(1424, 4)


,State,Population,Quarter,Quarter_End
0,NSW,5234889,1981Q2,1981-06-30
1,NSW,5249455,1981Q3,1981-09-30
2,NSW,5266894,1981Q4,1981-12-31
3,NSW,5286119,1982Q1,1982-03-31
4,NSW,5303580,1982Q2,1982-06-30


In [ ]:
sfd_raw = read_raw_sheet(path_for("sfd"), "Data1")

sfd_cols = select_state_columns(
    sfd_raw,
    description_keyword="STATE FINAL DEMAND",
    series_type="Seasonally Adjusted",
    exclude_keywords=["Percentage changes"],
)

sfd_long = wide_state_series_to_long(
    sfd_raw,
    selected_cols=sfd_cols,
    value_name="State_Final_Demand_SA",
)

print("Step 10 complete: State Final Demand extracted.")
print(sfd_long.shape)
sfd_long.head()

Step 10 complete: State Final Demand extracted.
(1296, 4)


,State,State_Final_Demand_SA,Quarter,Quarter_End
0,NSW,64918,1985Q3,1985-09-30
1,NSW,66372,1985Q4,1985-12-31
2,NSW,66018,1986Q1,1986-03-31
3,NSW,67903,1986Q2,1986-06-30
4,NSW,68205,1986Q3,1986-09-30


In [ ]:
unemployment_raw = read_raw_sheet(path_for("unemployment"), "Data2")
unemployment_meta = get_abs_metadata(unemployment_raw)

unemployment_cols = []

for abbr, full_name in STATE_NAMES.items():
    chosen_type = "Original" if abbr in ["NT", "ACT"] else "Seasonally Adjusted"

    mask = unemployment_meta["description"].str.contains(
        "Unemployment rate ; Persons",
        case=False,
        regex=False,
    )

    mask &= unemployment_meta["description"].str.contains(
        full_name,
        case=False,
        regex=False,
    )

    mask &= unemployment_meta["series_type"].eq(chosen_type)

    candidates = unemployment_meta[mask]

    if candidates.empty:
        raise ValueError(f"No unemployment column found for {abbr} using {chosen_type}")

    unemployment_cols.append((abbr, chosen_type, int(candidates.iloc[0]["col"])))

unemployment_qtr = monthly_state_series_to_quarter_average(
    unemployment_raw,
    selected_cols=unemployment_cols,
    value_name="Unemployment_Rate_QtrAvg",
)

print("Step 11 complete: Unemployment extracted and converted to quarterly average.")
print(unemployment_qtr.shape)
unemployment_qtr.head()

Step 11 complete: Unemployment extracted and converted to quarterly average.
(1544, 5)


,Quarter,Quarter_End,State,Unemployment_Rate_QtrAvg,Unemployment_Series_Type
0,1978Q1,1978-03-31,ACT,6.851605,Original
1,1978Q2,1978-06-30,ACT,6.132413,Original
2,1978Q3,1978-09-30,ACT,5.886824,Original
3,1978Q4,1978-12-31,ACT,7.844376,Original
4,1979Q1,1979-03-31,ACT,8.787887,Original


In [ ]:
cpi_raw = read_raw_sheet(path_for("cpi"), "Data1")

cpi_col = select_national_column(
    cpi_raw,
    required_keywords=["Index Numbers", "All groups CPI", "Australia"],
    series_type="Original",
)

cpi_qtr = wide_national_series_to_quarterly(
    cpi_raw,
    selected_col=cpi_col,
    value_name="CPI_All_Groups_Index",
)

print("Step 12 complete: CPI extracted.")
print(cpi_qtr.shape)
cpi_qtr.head()

Step 12 complete: CPI extracted.
(310, 3)


,CPI_All_Groups_Index,Quarter,Quarter_End
10,2.59,1948Q3,1948-09-30
11,2.63,1948Q4,1948-12-31
12,2.71,1949Q1,1949-03-31
13,2.74,1949Q2,1949-06-30
14,2.82,1949Q3,1949-09-30


In [ ]:
cash_raw = read_raw_sheet(path_for("cash_rate"), "Data")

cash_col = find_rba_cash_rate_column(cash_raw)

cash_qtr = monthly_national_to_quarter_average(
    cash_raw,
    selected_col=cash_col,
    value_name="Cash_Rate_QtrAvg",
)

print("Step 13 complete: Cash rate extracted and converted to quarterly average.")
print(cash_qtr.shape)
cash_qtr.head()

Step 13 complete: Cash rate extracted and converted to quarterly average.
(228, 3)


,Quarter,Quarter_End,Cash_Rate_QtrAvg
0,1969Q2,1969-06-30,NaN
1,1969Q3,1969-09-30,NaN
2,1969Q4,1969-12-31,NaN
3,1970Q1,1970-03-31,NaN
4,1970Q2,1970-06-30,NaN


In [ ]:
wpi_raw = read_raw_sheet(path_for("wpi"), "Data1")

wpi_col = select_national_column(
    wpi_raw,
    required_keywords=[
        "Quarterly Index",
        "Total hourly rates of pay excluding bonuses",
        "Australia",
        "Private and Public",
        "All industries",
    ],
    series_type="Seasonally Adjusted",
)

wpi_qtr = wide_national_series_to_quarterly(
    wpi_raw,
    selected_col=wpi_col,
    value_name="WPI_Index_SA",
)

print("Step 14 complete: WPI extracted.")
print(wpi_qtr.shape)
wpi_qtr.head()

Step 14 complete: WPI extracted.
(114, 3)


,WPI_Index_SA,Quarter,Quarter_End
10,66.6,1997Q3,1997-09-30
11,67.2,1997Q4,1997-12-31
12,67.8,1998Q1,1998-03-31
13,68.3,1998Q2,1998-06-30
14,68.8,1998Q3,1998-09-30


In [ ]:
household_raw = read_raw_sheet(path_for("household_spending"), "Data1")

household_states = {
    key: STATE_NAMES[key]
    for key in ["NSW", "VIC", "QLD", "SA", "WA", "TAS"]
}

household_cols = select_state_columns(
    household_raw,
    description_keyword="Total (Household Spending Categories)",
    series_type="Original",
    states=household_states,
    extra_required_keywords=["Current Price"],
)

household_long = wide_state_series_to_long(
    household_raw,
    selected_cols=household_cols,
    value_name="Household_Spending_QoQ_Pct",
)

print("Step 15 complete: Household spending extracted.")
print(household_long.shape)
household_long.head()

Step 15 complete: Household spending extracted.
(330, 4)


,State,Household_Spending_QoQ_Pct,Quarter,Quarter_End
0,NSW,NaN,2012Q3,2012-09-30
1,NSW,8.1,2012Q4,2012-12-31
2,NSW,-9.3,2013Q1,2013-03-31
3,NSW,2.1,2013Q2,2013-06-30
4,NSW,3.7,2013Q3,2013-09-30


In [ ]:
panel = retail_long.merge(
    population_long,
    on=["Quarter", "Quarter_End", "State"],
    how="inner",
)

panel = panel.merge(
    sfd_long,
    on=["Quarter", "Quarter_End", "State"],
    how="inner",
)

panel = panel.merge(
    unemployment_qtr,
    on=["Quarter", "Quarter_End", "State"],
    how="inner",
)

panel = panel.merge(
    cpi_qtr,
    on=["Quarter", "Quarter_End"],
    how="inner",
)

panel = panel.merge(
    cash_qtr,
    on=["Quarter", "Quarter_End"],
    how="inner",
)

panel = panel.merge(
    wpi_qtr,
    on=["Quarter", "Quarter_End"],
    how="inner",
)

panel = panel.merge(
    household_long,
    on=["Quarter", "Quarter_End", "State"],
    how="left",
)

print("Step 16 complete: All datasets merged.")
print(panel.shape)
panel.head()

Step 16 complete: All datasets merged.
(896, 12)


,State,Retail_Turnover_CVM_SA,Quarter,Quarter_End,Population,State_Final_Demand_SA,Unemployment_Rate_QtrAvg,Unemployment_Series_Type,CPI_All_Groups_Index,Cash_Rate_QtrAvg,WPI_Index_SA,Household_Spending_QoQ_Pct
0,NSW,15648.9,1997Q3,1997-09-30,6260867,94122,7.759386,Seasonally Adjusted,46.28,5.152174,66.6,NaN
1,NSW,15804.2,1997Q4,1997-12-31,6274966,97053,7.453965,Seasonally Adjusted,46.39,5.000000,67.2,NaN
2,NSW,15791.8,1998Q1,1998-03-31,6295260,97387,7.257086,Seasonally Adjusted,46.51,5.000000,67.8,NaN
3,NSW,15699.5,1998Q2,1998-06-30,6305799,97546,7.156576,Seasonally Adjusted,46.78,5.000000,68.3,NaN
4,NSW,15654.9,1998Q3,1998-09-30,6324111,100036,7.154846,Seasonally Adjusted,46.91,5.000000,68.8,NaN


In [ ]:
PANEL_START = pd.Period("1997Q3", freq="Q")
PANEL_END = pd.Period("2025Q2", freq="Q")

panel["_period"] = pd.PeriodIndex(panel["Quarter"], freq="Q")

panel = panel[
    (panel["_period"] >= PANEL_START)
    & (panel["_period"] <= PANEL_END)
].copy()

print("Step 17 complete: Period restricted.")
print(panel["Quarter"].min(), "to", panel["Quarter"].max())
print(panel.shape)

Step 17 complete: Period restricted.
1997Q3 to 2025Q2
(896, 13)


In [ ]:
panel["State"] = pd.Categorical(
    panel["State"],
    categories=STATE_ORDER,
    ordered=True,
)

panel = panel.sort_values(["State", "Quarter_End"]).reset_index(drop=True)

panel["Retail_Turnover_QoQ_Pct"] = (
    panel.groupby("State", observed=False)["Retail_Turnover_CVM_SA"]
    .pct_change()
    * 100
)

panel["Retail_Turnover_Per_Capita"] = (
    panel["Retail_Turnover_CVM_SA"] / panel["Population"]
)

panel["Unemployment_QoQ_Pct"] = (
    panel.groupby("State", observed=False)["Unemployment_Rate_QtrAvg"]
    .pct_change()
    * 100
)

panel["CPI_QoQ_Pct"] = (
    panel.groupby("State", observed=False)["CPI_All_Groups_Index"]
    .pct_change()
    * 100
)

panel["Cash_Rate_QoQ_Pct"] = (
    panel.groupby("State", observed=False)["Cash_Rate_QtrAvg"]
    .pct_change()
    * 100
)

panel["WPI_QoQ_Pct"] = (
    panel.groupby("State", observed=False)["WPI_Index_SA"]
    .pct_change()
    * 100
)

panel["State_Final_Demand_QoQ_Pct"] = (
    panel.groupby("State", observed=False)["State_Final_Demand_SA"]
    .pct_change()
    * 100
)

panel["Population_QoQ_Pct"] = (
    panel.groupby("State", observed=False)["Population"]
    .pct_change()
    * 100
)

print("Step 18 complete: Derived variables calculated.")
panel.head()

Step 18 complete: Derived variables calculated.


,State,Retail_Turnover_CVM_SA,Quarter,Quarter_End,Population,State_Final_Demand_SA,Unemployment_Rate_QtrAvg,Unemployment_Series_Type,CPI_All_Groups_Index,Cash_Rate_QtrAvg,...,Household_Spending_QoQ_Pct,_period,Retail_Turnover_QoQ_Pct,Retail_Turnover_Per_Capita,Unemployment_QoQ_Pct,CPI_QoQ_Pct,Cash_Rate_QoQ_Pct,WPI_QoQ_Pct,State_Final_Demand_QoQ_Pct,Population_QoQ_Pct
0,ACT,856.0,1997Q3,1997-09-30,310258,7199,7.228657,Original,46.28,5.152174,...,NaN,1997Q3,NaN,0.002759,NaN,NaN,NaN,NaN,NaN,NaN
1,ACT,862.2,1997Q4,1997-12-31,310281,7399,7.204838,Original,46.39,5.000000,...,NaN,1997Q4,0.724299,0.002779,-0.329512,0.237684,-2.953586,0.900901,2.778164,0.007413
2,ACT,861.8,1998Q1,1998-03-31,311026,7257,8.153327,Original,46.51,5.000000,...,NaN,1998Q1,-0.046393,0.002771,13.164614,0.258676,0.000000,0.892857,-1.919178,0.240105
3,ACT,852.4,1998Q2,1998-06-30,311532,7114,6.556750,Original,46.78,5.000000,...,NaN,1998Q2,-1.090740,0.002736,-19.581905,0.580520,0.000000,0.737463,-1.970511,0.162687
4,ACT,866.3,1998Q3,1998-09-30,311732,7578,6.374642,Original,46.91,5.000000,...,NaN,1998Q3,1.630690,0.002779,-2.777407,0.277897,0.000000,0.732064,6.522350,0.064199


In [ ]:
OUTPUT_COLUMNS = [
    "Quarter",
    "Quarter_End",
    "State",
    "Retail_Turnover_CVM_SA",
    "Retail_Turnover_QoQ_Pct",
    "Retail_Turnover_Per_Capita",
    "Unemployment_Rate_QtrAvg",
    "Unemployment_Series_Type",
    "Unemployment_QoQ_Pct",
    "CPI_All_Groups_Index",
    "CPI_QoQ_Pct",
    "Cash_Rate_QtrAvg",
    "Cash_Rate_QoQ_Pct",
    "WPI_Index_SA",
    "WPI_QoQ_Pct",
    "State_Final_Demand_SA",
    "State_Final_Demand_QoQ_Pct",
    "Population",
    "Population_QoQ_Pct",
    "Household_Spending_QoQ_Pct",
]

full_history = panel[OUTPUT_COLUMNS].copy()

print("Step 19 complete: Full-history table ready.")
print(full_history.shape)
full_history.head()

Step 19 complete: Full-history table ready.
(896, 20)


,Quarter,Quarter_End,State,Retail_Turnover_CVM_SA,Retail_Turnover_QoQ_Pct,Retail_Turnover_Per_Capita,Unemployment_Rate_QtrAvg,Unemployment_Series_Type,Unemployment_QoQ_Pct,CPI_All_Groups_Index,CPI_QoQ_Pct,Cash_Rate_QtrAvg,Cash_Rate_QoQ_Pct,WPI_Index_SA,WPI_QoQ_Pct,State_Final_Demand_SA,State_Final_Demand_QoQ_Pct,Population,Population_QoQ_Pct,Household_Spending_QoQ_Pct
0,1997Q3,1997-09-30,ACT,856.0,NaN,0.002759,7.228657,Original,NaN,46.28,NaN,5.152174,NaN,66.6,NaN,7199,NaN,310258,NaN,NaN
1,1997Q4,1997-12-31,ACT,862.2,0.724299,0.002779,7.204838,Original,-0.329512,46.39,0.237684,5.000000,-2.953586,67.2,0.900901,7399,2.778164,310281,0.007413,NaN
2,1998Q1,1998-03-31,ACT,861.8,-0.046393,0.002771,8.153327,Original,13.164614,46.51,0.258676,5.000000,0.000000,67.8,0.892857,7257,-1.919178,311026,0.240105,NaN
3,1998Q2,1998-06-30,ACT,852.4,-1.090740,0.002736,6.556750,Original,-19.581905,46.78,0.580520,5.000000,0.000000,68.3,0.737463,7114,-1.970511,311532,0.162687,NaN
4,1998Q3,1998-09-30,ACT,866.3,1.630690,0.002779,6.374642,Original,-2.777407,46.91,0.277897,5.000000,0.000000,68.8,0.732064,7578,6.522350,311732,0.064199,NaN


In [ ]:
overlap = full_history.copy()
overlap["_period"] = pd.PeriodIndex(overlap["Quarter"], freq="Q")

overlap = overlap[
    (overlap["State"].astype(str).isin(HS_STATE_ORDER))
    & (overlap["_period"] >= pd.Period("2012Q4", freq="Q"))
    & (overlap["_period"] <= pd.Period("2025Q2", freq="Q"))
].drop(columns=["_period"]).copy()

overlap["State"] = pd.Categorical(
    overlap["State"].astype(str),
    categories=HS_STATE_ORDER,
    ordered=True,
)

overlap = overlap.sort_values(["State", "Quarter_End"]).reset_index(drop=True)

print("Step 20 complete: Household-spending overlap table ready.")
print(overlap.shape)
overlap.head()

Step 20 complete: Household-spending overlap table ready.
(306, 20)


,Quarter,Quarter_End,State,Retail_Turnover_CVM_SA,Retail_Turnover_QoQ_Pct,Retail_Turnover_Per_Capita,Unemployment_Rate_QtrAvg,Unemployment_Series_Type,Unemployment_QoQ_Pct,CPI_All_Groups_Index,CPI_QoQ_Pct,Cash_Rate_QtrAvg,Cash_Rate_QoQ_Pct,WPI_Index_SA,WPI_QoQ_Pct,State_Final_Demand_SA,State_Final_Demand_QoQ_Pct,Population,Population_QoQ_Pct,Household_Spending_QoQ_Pct
0,2012Q4,2012-12-31,NSW,24266.3,-0.025131,0.003300,5.098420,Seasonally Adjusted,0.970650,70.87,0.254633,3.183333,-9.047619,114.2,0.705467,153234,1.073169,7353189,0.349022,8.1
1,2013Q1,2013-03-31,NSW,24754.3,2.011019,0.003353,5.257358,Seasonally Adjusted,3.117398,71.11,0.338648,3.000000,-5.759162,115.0,0.700525,152425,-0.527951,7382957,0.404831,-9.3
2,2013Q2,2013-06-30,NSW,24795.7,0.167244,0.003349,5.500782,Seasonally Adjusted,4.630172,71.38,0.379693,2.850000,-5.000000,115.7,0.608696,151524,-0.591110,7404032,0.285455,2.1
3,2013Q3,2013-09-30,NSW,25003.1,0.836435,0.003365,5.720961,Seasonally Adjusted,4.002672,72.25,1.218829,2.600000,-8.771930,116.4,0.605013,152789,0.834851,7430904,0.362937,3.7
4,2013Q4,2013-12-31,NSW,25470.8,1.870568,0.003417,5.900415,Seasonally Adjusted,3.136788,72.77,0.719723,2.500000,-3.846154,117.2,0.687285,154136,0.881608,7454938,0.323433,8.6


In [35]:
readme = pd.DataFrame([
    ["Merged quarterly regression dataset rebuilt with long-run population and household spending"],
    [""],
    ["Main sheet"],
    ["State_Qtr_AllStates_FullHistory: 1997Q3 to 2025Q2, all 8 states/territories, core variables + full-history population + household spending where available."],
    [""],
    ["Important notes"],
    ["1. Population is now sourced from 310104.xlsx (Estimated Resident Population ; Persons ; <State>) and is available through the full panel period used here."],
    ["2. Household spending comes from 5682014.xlsx and is a quarterly percentage-change series for Total (Household Spending Categories), Current Price, Original series."],
    ["3. Household spending is only available for NSW, VIC, QLD, SA, WA and TAS, and only from 2012Q4 onward in this merged panel."],
    ["4. NT and ACT do not have household spending values in the uploaded 5682014.xlsx workbook, so those cells are blank in the full-history sheet."],
    ["5. For unemployment, NSW/VIC/QLD/SA/WA/TAS use Seasonally Adjusted series; NT and ACT use Original series because SA was not available in the uploaded labour-force workbook."],
    ["6. QoQ percentage-change columns are blank in each state's first observation because no previous quarter exists for comparison."],
    [""],
    ["Secondary sheet"],
    ["State_Qtr_6States_HS_Overlap: 2012Q4 to 2025Q2, only NSW/VIC/QLD/SA/WA/TAS where household spending is available."]
])

source_mapping = pd.DataFrame([
    [
        "Output_Variable",
        "Source_File",
        "Sheet",
        "Series_Description",
        "Series_Type_Used",
        "Availability / Notes",
    ],
    [
        "Retail_Turnover_CVM_SA",
        "850109.xlsx",
        "Data1",
        "Turnover - Chain Volume Measure ; <State> ; Total (Industry) ;",
        "Seasonally Adjusted",
        "All 8 states/territories, 1983Q3 to 2025Q2 in source; used here 1997Q3 to 2025Q2",
    ],
    [
        "Unemployment_Rate_QtrAvg",
        "6202012.xlsx",
        "Data2",
        "Unemployment rate ; Persons ; <State>",
        "SA for NSW/VIC/QLD/SA/WA/TAS; Original for NT/ACT",
        "Monthly source averaged to quarter",
    ],
    [
        "CPI_All_Groups_Index",
        "6401017.xlsx",
        "Data1",
        "Index Numbers ; All groups CPI ; Australia ;",
        "Original",
        "National quarterly series",
    ],
    [
        "Cash_Rate_QtrAvg",
        "f01hist.xlsx",
        "Data",
        "Cash Rate Target",
        "Monthly average to quarterly average",
        "National series",
    ],
    [
        "WPI_Index_SA",
        "634501.xlsx",
        "Data1",
        "Quarterly Index ; Total hourly rates of pay excluding bonuses ; Australia ; Private and Public ; All industries ;",
        "Seasonally Adjusted",
        "National quarterly series; drives full panel start at 1997Q3",
    ],
    [
        "State_Final_Demand_SA",
        "5206025_SFD_Summary.xlsx",
        "Data1",
        "<State> ; STATE FINAL DEMAND ;",
        "Seasonally Adjusted",
        "All 8 states/territories",
    ],
    [
        "Population",
        "310104.xlsx",
        "Data1",
        "Estimated Resident Population ; Persons ; <State> ;",
        "Original",
        "All 8 states/territories; long-run quarterly history",
    ],
    [
        "Household_Spending_QoQ_Pct",
        "5682014.xlsx",
        "Data1",
        "Household spending - Percentage change from previous period ; Total (Household Spending Categories) ; <State> ; Current Price ;",
        "Original",
        "Only NSW/VIC/QLD/SA/WA/TAS; first usable value 2012Q4",
    ],
])

print("Step 21 complete: README and Source Mapping created in File 1 format.")

Step 21 complete: README and Source Mapping created in File 1 format.


In [36]:
output_path = os.path.join(RAW_FOLDER, OUTPUT_FILE)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # Sheet 1: README
    # header=False is important here.
    # It stops pandas from applying header style to A1.
    readme.to_excel(
        writer,
        sheet_name="README",
        index=False,
        header=False
    )

    # Sheet 2: Source_Mapping
    # Keep header=True/default to create the extra first row 0,1,2,3,4,5 like File 1.
    source_mapping.to_excel(
        writer,
        sheet_name="Source_Mapping",
        index=False
    )

    # Sheet 3: Full history
    full_history.to_excel(
        writer,
        sheet_name="State_Qtr_AllStates_FullHistory",
        index=False,
        startrow=2
    )

    # Sheet 4: Household spending overlap
    overlap.to_excel(
        writer,
        sheet_name="State_Qtr_6States_HS_Overlap",
        index=False,
        startrow=2
    )

    workbook = writer.book

    ws_full = workbook["State_Qtr_AllStates_FullHistory"]
    ws_overlap = workbook["State_Qtr_6States_HS_Overlap"]

    # Add the same title cells as File 1
    ws_full["A1"] = "All-states quarterly merged panel, full history (1997Q3-2025Q2)"
    ws_overlap["A1"] = "Six-state quarterly panel with household spending available (2012Q4-2025Q2)"

    # IMPORTANT:
    # File 1 physically stores blank cells in A2:T2.
    # This makes row 2 not just visually blank, but structurally present.
    for col in range(1, 21):
        ws_full.cell(row=2, column=col, value="")
        ws_overlap.cell(row=2, column=col, value="")

print("Step 22 complete: File 1-style Excel file exported.")
print(output_path)

Step 22 complete: File 1-style Excel file exported.
/content/quarterly_regression_merged_dataset_rebuilt_fullhistory_pop_hs.xlsx


In [33]:
print("MAIN OUTPUT CHECKS")
print("-" * 50)

print("Full-history panel shape:", full_history.shape)
print("Expected full-history shape: (896, 20)")
print("Full-history period:", full_history["Quarter"].min(), "to", full_history["Quarter"].max())
print("Full-history states:", list(full_history["State"].astype(str).unique()))

print()
print("HOUSEHOLD-SPENDING OVERLAP CHECKS")
print("-" * 50)

print("Overlap panel shape:", overlap.shape)
print("Expected overlap shape: (306, 20)")
print("Overlap period:", overlap["Quarter"].min(), "to", overlap["Quarter"].max())
print("Overlap states:", list(overlap["State"].astype(str).unique()))

print()
print("Final output file location:")
print(output_path)

MAIN OUTPUT CHECKS
--------------------------------------------------
Full-history panel shape: (896, 20)
Expected full-history shape: (896, 20)
Full-history period: 1997Q3 to 2025Q2
Full-history states: ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

HOUSEHOLD-SPENDING OVERLAP CHECKS
--------------------------------------------------
Overlap panel shape: (306, 20)
Expected overlap shape: (306, 20)
Overlap period: 2012Q4 to 2025Q2
Overlap states: ['NSW', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

Final output file location:
/content/quarterly_regression_merged_dataset_rebuilt_fullhistory_pop_hs.xlsx


In [37]:
from google.colab import files

files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>